# PR #9 validation: backend vs training preprocessing, full API test

Runs the **backend code from the `feat/model-integration` branch** with the **trained model** on real FF++ C40 test videos.

1. **Parity:** frame indices, face detected per frame, face boxes, all 32 crops, the 16 crops given to the model, the final tensor (JPEG + ImageNet normalisation), sequence length, time order. Each is compared against the training preprocessing (functions copied unchanged from `preprocessing/FFpp_c40_Preprocessing.ipynb`) and against the training JPEGs saved in `faces.tar`.
2. **Live API:** starts the backend the normal way (uvicorn, model loaded at startup) and POSTs each video to `/api/v1/analysis`. It checks:
   - verdict and probabilities
   - `NO_FACE` for a video without a face
   - videos where the face is missing in some frames
   - Grad-CAM image (same as the in-process run of the same video)

**Needs:** a GPU runtime (T4), plus on Drive `FFpp_c40/` (raw videos), `FFpp_c40_processed/` (`manifest.csv`, `faces.tar`) and `deepguard_v1.pt`.
Results are saved to `MyDrive/DeepGuard_runs/pr9_validation/`.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name --format=csv,noheader
!pip install -q fastapi uvicorn python-multipart pydantic-settings
!pip install -q facenet-pytorch==2.6.0 --no-deps   # --no-deps keeps Colab's torch untouched

In [ ]:
# 2. Config
from pathlib import Path

REPO   = 'https://github.com/praveen24sriv/Deepfake-video-detection-system.git'
BRANCH = 'feat/model-integration'
DRIVE_SRC  = Path('/content/drive/MyDrive/FFpp_c40')              # raw videos: <method>/<video_id>.mp4
DRIVE_PROC = Path('/content/drive/MyDrive/FFpp_c40_processed')    # manifest.csv + faces.tar
OUT        = Path('/content/drive/MyDrive/DeepGuard_runs/pr9_validation')

# model file: first match wins
candidates = [Path('/content/drive/MyDrive/DeepGuard_runs/cnn_lstm/deepguard_v1.pt'),
              Path('/content/drive/MyDrive/deepguard_v1.pt')]
MODEL = next((p for p in candidates if p.exists()), None)
if MODEL is None:
    found = sorted(Path('/content/drive/MyDrive').glob('**/deepguard_v1.pt'))
    MODEL = found[0] if found else None
assert MODEL is not None, 'deepguard_v1.pt not found on Drive: set MODEL manually'
print('model :', MODEL)

# test videos (official FF++ test split): same 6 as the first check, fakes from all 4 trained methods
REAL_IDS = ['003', '012', '015']
FAKE_IDS = ['003_000', '012_026', '015_919']
METHODS  = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']

In [ ]:
# 3. Get the PR branch (the exact code under review)
!rm -rf /content/repo && git clone -q --depth 1 -b {BRANCH} {REPO} /content/repo
!cd /content/repo && git log -1 --format='commit %h  %ad  %s' --date=short
src = Path('/content/repo/backend/app/services/analysis_service.py').read_text()
assert 'FaceCropper' in src, 'analysis_service.py on the branch is the OLD version: upload the fixed file first'
print('fixed analysis_service.py found on the branch')

In [ ]:
%%writefile /content/parity_check.py
"""
DeepGuard PR #9 validation: training vs backend preprocessing parity + live API checks.

Usage (from any folder):
    python parity_check.py --backend <repo>/backend --model <path>/deepguard_v1.pt \
        --videos <labels.csv> [--faces <dir with training crops>] [--out results]

labels.csv columns: path, label (REAL / FAKE / NO_FACE), crops_dir (optional, relative to --faces)

What it checks for every video:
  1. Parity (in-process, the backend's own code): frame indices, face detected per frame,
     face boxes, all 32 crops, the 16 selected crops, the final model tensor (after JPEG +
     normalisation), sequence length and time order -- against the TRAINING preprocessing
     (functions copied from preprocessing/FFpp_c40_Preprocessing.ipynb).
  2. Optional: backend crops vs the actual training JPEGs saved by the preprocessing notebook.
  3. Live API: starts the real backend (uvicorn, normal startup path), POSTs each video to
     /api/v1/analysis, checks verdict / probabilities / NO_FACE / Grad-CAM image.
"""
import argparse
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------------ training reference
# Copied from preprocessing/FFpp_c40_Preprocessing.ipynb (cell 5), unchanged.
N_FRAMES, CROP_SIZE, MARGIN = 32, 224, 1.3
mtcnn = None


def ref_sample_frames(path, n=N_FRAMES):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return [], []
    wanted = set(np.linspace(0, total - 1, num=min(n, total)).round().astype(int).tolist())
    frames, idxs, i = [], [], 0
    while True:
        ok = cap.grab()
        if not ok: break
        if i in wanted:
            ok, f = cap.retrieve()
            if ok:
                frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)); idxs.append(i)
        i += 1
        if i > max(wanted): break
    cap.release()
    return frames, idxs


def ref_square_box(box, w, h, margin=MARGIN):
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * margin
    side = min(side, w, h)
    x1 = int(np.clip(cx - side / 2, 0, w - side)); y1 = int(np.clip(cy - side / 2, 0, h - side))
    return x1, y1, x1 + int(side), y1 + int(side)


def ref_largest(boxes):
    if boxes is None or len(boxes) == 0: return None
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return boxes[int(np.argmax(areas))]


def ref_crop_video(frames):
    """Training crop_video, plus the per-frame box used (for the parity report)."""
    h, w = frames[0].shape[:2]
    boxes, _ = mtcnn.detect(frames)
    crops, used, detected, last = [], [], [], None
    for f, b in zip(frames, boxes):
        b = ref_largest(b)
        detected.append(b is not None)
        if b is not None:
            last = b
        if last is None:
            continue
        x1, y1, x2, y2 = ref_square_box(last, w, h)
        used.append((x1, y1, x2, y2))
        crops.append(cv2.resize(f[y1:y2, x1:x2], (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_AREA))
    if crops and len(crops) < len(frames):
        pad = len(frames) - len(crops)
        crops = [crops[0]] * pad + crops
        used = [used[0]] * pad + used
    return crops, used, detected


def ref_select(items, t=16):
    """Training evaluation frame choice: frame_indices(n, 16, train=False)."""
    n = len(items)
    idx = np.clip(np.round(np.linspace(0, n - 1, t)), 0, n - 1).astype(int)
    return [items[i] for i in idx]


# ------------------------------------------------------------------ helpers
def max_diff(a, b):
    return float(np.abs(np.asarray(a, np.float64) - np.asarray(b, np.float64)).max())


def backend_trace(analysis_service, detector, video_service, video):
    """Run the backend's analyze() in-process and record what each stage produced."""
    cap = {}
    cropper = getattr(analysis_service, "face_cropper", None)
    if cropper is None:          # old backend code (before the fix): no shared cropper to inspect
        class _Null: pass
        cropper = _Null(); cropper.mtcnn = _Null()
        cropper.mtcnn.detect = cropper.crop = None
    det_orig, crop_orig = cropper.mtcnn.detect, cropper.crop

    def det_spy(frames, *a, **k):
        boxes, probs = det_orig(frames, *a, **k)
        cap["boxes"] = boxes
        return boxes, probs

    def crop_spy(frames):
        crops, detected = crop_orig(frames)
        cap["crops32"], cap["detected"] = [c.copy() for c in crops], list(detected)
        return crops, detected

    if crop_orig is not None:
        cropper.mtcnn.detect, cropper.crop = det_spy, crop_spy
    pc_orig = None
    if detector.available:
        pc_orig = detector.predictor.predict_crops

        def pc_spy(crops, ts, heatmap_path=None):
            cap["model_crops"], cap["model_ts"] = [c.copy() for c in crops], list(ts)
            return pc_orig(crops, ts, heatmap_path=heatmap_path)

        detector.predictor.predict_crops = pc_spy
    try:
        resp = analysis_service.analyze(video, Path(video).name)
    finally:
        cropper.mtcnn.detect, cropper.crop = det_orig, crop_orig
        if pc_orig is not None:
            detector.predictor.predict_crops = pc_orig
    cap["indices"] = [s.index for s in video_service.inspect(video).samples]
    cap["response"] = resp
    return cap


def parity_row(name, video, bt, dg, faces_dir=None, crops_dir=None):
    frames, idxs = ref_sample_frames(video)
    r = dict(video=name)
    r["frame_indices_match"] = idxs == bt["indices"]
    r["n_sampled"] = len(idxs)
    if not frames:
        return r
    ref_crops, ref_boxes, ref_det = ref_crop_video(frames)
    r["faces_detected"] = int(sum(ref_det))
    r["detected_flags_match"] = (ref_det == bt["detected"]) if "detected" in bt else "n/a"
    # face boxes the backend's MTCNN returned vs the training MTCNN (largest face per frame)
    diffs = []
    for a, b in zip(mtcnn.detect(frames)[0], bt.get("boxes", [])):
        a, b = ref_largest(a), ref_largest(b)
        if (a is None) != (b is None):
            diffs.append(np.inf)
        elif a is not None:
            diffs.append(max_diff(a, b))
    r["box_max_diff_px"] = round(max(diffs), 3) if diffs else None
    if not ref_crops:
        r["both_no_face"] = not bt.get("crops32") and bt["response"].detection.verdict == "NO_FACE"
        return r
    if bt.get("crops32"):
        r["crops32_max_pixel_diff"] = max_diff(np.stack(ref_crops), np.stack(bt["crops32"]))
    ref16 = ref_select(ref_crops)
    if "model_crops" in bt:
        r["seq_len"] = len(bt["model_crops"])
        r["crop_shape"] = "x".join(map(str, bt["model_crops"][0].shape))
        be16 = [dg.resample(bt["model_crops"], 16) if len(bt["model_crops"]) != 16 else bt["model_crops"]][0]
        r["crops16_max_pixel_diff"] = max_diff(np.stack(ref16), np.stack(be16))
        r["crops16_mean_pixel_diff"] = round(float(np.abs(np.stack(ref16).astype(float) - np.stack(be16)).mean()), 3)
        t_ref = dg.to_tensor([dg.as_training_jpeg(c) for c in ref16])
        t_be = dg.to_tensor([dg.as_training_jpeg(c) for c in be16])
        r["tensor_shape"] = "x".join(map(str, t_be.shape))
        r["tensor_max_diff"] = float((t_ref - t_be).abs().max())
        r["time_order_ok"] = all(b > a for a, b in zip(bt["model_ts"], bt["model_ts"][1:]))
        ref_ts = ref_select([i / cv2.VideoCapture(str(video)).get(cv2.CAP_PROP_FPS) for i in idxs])
        be_ts = bt["model_ts"] if len(bt["model_ts"]) == 16 else dg.resample(bt["model_ts"], 16)
        r["timestamps_match"] = bool(np.allclose(ref_ts, be_ts))
    if faces_dir and crops_dir:
        folder = Path(faces_dir) / crops_dir
        saved = sorted(folder.glob("*.jpg"))
        if saved:
            train32 = [cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB) for p in saved]
            be32 = [dg.as_training_jpeg(c) for c in bt["crops32"]]
            if len(train32) == len(be32):
                d = np.abs(np.stack(train32).astype(np.int16) - np.stack(be32).astype(np.int16))
                r["vs_training_jpegs_mean_diff"] = round(float(d.mean()), 3)
                r["vs_training_jpegs_max_diff"] = int(d.max())
            else:
                r["vs_training_jpegs_mean_diff"] = f"count {len(train32)} vs {len(be32)}"
    return r


# ------------------------------------------------------------------ API
def start_api(backend, model, port, log):
    env = dict(os.environ, DEMO_MODE="false", MODEL_PATH=str(model), MODEL_DEVICE="")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", str(port)],
        cwd=backend, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    import requests
    for _ in range(240):
        if proc.poll() is not None:
            raise RuntimeError("backend exited during startup, see " + str(log))
        try:
            if requests.get(f"http://127.0.0.1:{port}/api/v1/health", timeout=2).ok:
                return proc
        except requests.RequestException:
            pass
        time.sleep(1)
    proc.kill()
    raise RuntimeError("backend did not start in 240 s, see " + str(log))


def api_row(port, name, video, label, out_dir):
    import requests
    base = f"http://127.0.0.1:{port}"
    t0 = time.time()
    with open(video, "rb") as fh:
        res = requests.post(f"{base}/api/v1/analysis", files={"file": (Path(video).name, fh, "video/mp4")},
                            timeout=600)
    r = dict(video=name, expected=label, http=res.status_code, api_seconds=round(time.time() - t0, 1))
    if not res.ok:
        r["error"] = res.text[:200]
        return r, None
    j = res.json()
    d, p, m = j["detection"], j["preprocessing"], j["model"]
    fake_p = d["confidence"] if d["verdict"] == "FAKE" else round(1 - d["confidence"], 4) if d["verdict"] == "REAL" else None
    r.update(verdict=d["verdict"], confidence=d["confidence"], fake_prob=fake_p, correct=d["verdict"] == label,
             faces_detected=p["faces_detected"], frames_without_face=p["frames_without_face"],
             model_frames_used=p["model_frames_used"], demo_mode=m["demo_mode"],
             suspicious=f'{d["timeline"]["suspicious_start"]}-{d["timeline"]["suspicious_end"]}')
    heat = None
    url = j["meta"]["heatmap_url"]
    if url:
        hr = requests.get(base + url, timeout=30)
        img = cv2.imdecode(np.frombuffer(hr.content, np.uint8), cv2.IMREAD_COLOR) if hr.ok else None
        r["gradcam"] = "ok" if img is not None and img.shape[:2] == (224, 224) else "bad"
        if img is not None:
            heat = img
            cv2.imwrite(str(out_dir / f"gradcam_{name}.jpg"), img)
    else:
        r["gradcam"] = "none"
    return r, heat


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--backend", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--videos", required=True)
    ap.add_argument("--faces", default=None)
    ap.add_argument("--out", default="results")
    ap.add_argument("--port", type=int, default=8765)
    a = ap.parse_args()

    backend, out = Path(a.backend).resolve(), Path(a.out).resolve()
    out.mkdir(parents=True, exist_ok=True)
    vids = pd.read_csv(a.videos, dtype=str).fillna("")

    # backend code, in-process (same modules uvicorn will load)
    os.environ.update(DEMO_MODE="false", MODEL_PATH=str(Path(a.model).resolve()), MODEL_DEVICE="")
    cwd = os.getcwd(); os.chdir(backend); sys.path.insert(0, str(backend))
    from app.services.analysis_service import analysis_service
    from app.services.detector import detector
    from app.services.video_service import video_service
    from app.services import deepguard as dg
    from app.config import settings
    settings.output_dir.mkdir(parents=True, exist_ok=True)
    analysis_service.initialize()
    os.chdir(cwd)

    global mtcnn
    from facenet_pytorch import MTCNN
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    mtcnn = MTCNN(keep_all=True, device=dev, select_largest=True, post_process=False)   # as in training

    cfg = detector.predictor.cfg
    print(f"model: {cfg['version']} {cfg['run']} | test AUC {cfg['test_auc']} | threshold {cfg['threshold']} "
          f"| device {detector.predictor.device}")
    print(f"normalisation: checkpoint mean/std {tuple(cfg['mean'])}/{tuple(cfg['std'])} "
          f"| deepguard.py {dg.MEAN}/{dg.STD} | match: {tuple(cfg['mean']) == dg.MEAN and tuple(cfg['std']) == dg.STD}")

    def vname(v):
        return getattr(v, "video", "") or Path(v.path).stem

    def training_score(crops_dir):
        """Score from the saved training crops, as the test set was evaluated (16 evenly spaced)."""
        if not (a.faces and crops_dir):
            return None
        saved = sorted((Path(a.faces) / crops_dir).glob("*.jpg"))
        if not saved:
            return None
        crops = ref_select([cv2.cvtColor(cv2.imread(str(q)), cv2.COLOR_BGR2RGB) for q in saved])
        pr = detector.predictor
        with torch.no_grad():
            logit, _ = pr.model(dg.to_tensor(crops).unsqueeze(0).to(pr.device))
        return round(float(torch.sigmoid(logit - dg._logit(pr.threshold))[0]), 4)

    parity, inproc = [], {}
    for v in vids.itertuples():
        name = vname(v)
        bt = backend_trace(analysis_service, detector, video_service, v.path)
        resp = bt["response"]
        heat = None
        if resp.meta.heatmap_url:
            heat = cv2.imread(str(settings.output_dir / Path(resp.meta.heatmap_url).name))
        inproc[name] = (resp, heat, training_score(getattr(v, "crops_dir", "") or None))
        parity.append(parity_row(name, v.path, bt, dg, a.faces, getattr(v, "crops_dir", "") or None))
        print("parity", name, "done")
    parity = pd.DataFrame(parity)
    parity.to_csv(out / "parity.csv", index=False)

    proc = start_api(backend, Path(a.model).resolve(), a.port, out / "backend.log")
    try:
        rows = []
        for v in vids.itertuples():
            name = vname(v)
            r, heat = api_row(a.port, name, v.path, v.label, out)
            resp, ip_heat, tscore = inproc[name]
            r["inprocess_verdict"] = resp.detection.verdict
            r["inprocess_confidence"] = resp.detection.confidence
            if tscore is not None:
                r["training_crops_fake_prob"] = tscore
            if heat is not None and ip_heat is not None:
                r["gradcam_vs_inprocess_mean_diff"] = round(float(np.abs(heat.astype(float) - ip_heat).mean()), 2)
            rows.append(r)
            print("api", name, r.get("verdict"), r.get("confidence"))
    finally:
        proc.terminate()
    api = pd.DataFrame(rows)
    api.to_csv(out / "api_results.csv", index=False)

    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
    print("\n=== PARITY: training preprocessing vs backend ===")
    print(parity.to_string(index=False))
    print("\n=== LIVE API ===")
    print(api.to_string(index=False))
    with open(out / "summary.json", "w") as fh:
        json.dump(dict(model=cfg["version"], test_auc=cfg["test_auc"], threshold=cfg["threshold"],
                       n_videos=len(api), api_verdicts_correct=int(api.get("correct", pd.Series()).sum()),
                       torch=torch.__version__), fh, indent=1)
    print("\nsaved:", sorted(p.name for p in out.iterdir()))


if __name__ == "__main__":
    main()

In [ ]:
# 4. Test videos: real FF++ test videos + one with missing-face frames + one with no face
import shutil, cv2, numpy as np, pandas as pd

m = pd.read_csv(DRIVE_PROC / 'manifest.csv', dtype={'video_id': str})
VID = Path('/content/test_videos'); VID.mkdir(exist_ok=True)
rows = []
def add(method, vid, label):
    r = m[(m.method == method) & (m.video_id == vid)]
    src = DRIVE_SRC / method / f'{vid}.mp4'
    if r.empty or not src.exists():
        print('missing:', method, vid); return
    dst = VID / f'{method}_{vid}.mp4'; shutil.copy(src, dst)
    rows.append(dict(video=f'{method}_{vid}', path=str(dst), label=label, split=r.split.iloc[0],
                     crops_dir=r.crops_dir.iloc[0] if r.status.iloc[0] == 'ok' else ''))
for v in REAL_IDS: add('real', v, 'REAL')
for meth in METHODS:
    for v in FAKE_IDS: add(meth, v, 'FAKE')

def rewrite(src, dst, keep):
    """Re-encode src; frames where keep(i) is False become a plain grey frame (no face)."""
    cap = cv2.VideoCapture(str(src)); fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(3)), int(cap.get(4)); n = int(cap.get(7))
    vw = cv2.VideoWriter(str(dst), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for i in range(n):
        ok, f = cap.read()
        if not ok: break
        vw.write(f if keep(i, n) else np.full((h, w, 3), 90, np.uint8))
    vw.release(); cap.release()

# blank the frames at sampled positions 0-2 (start: back-fill), 12-15 and 25 (middle: reuse last box)
def partial_rule(i, n):
    wanted = np.linspace(0, n - 1, 32).round().astype(int)
    return i not in set(wanted[[0, 1, 2, 12, 13, 14, 15, 25]].tolist())

for method, vid, label in [('real', '003', 'REAL'), ('Deepfakes', '003_000', 'FAKE')]:
    src = VID / f'{method}_{vid}.mp4'
    if src.exists():
        dst = VID / f'partial_{method}_{vid}.mp4'; rewrite(src, dst, partial_rule)
        rows.append(dict(video=dst.stem, path=str(dst), label=label, split='test (edited)', crops_dir=''))
nf = VID / 'noface_grey.mp4'; rewrite(VID / 'real_003.mp4', nf, lambda i, n: False)
rows.append(dict(video=nf.stem, path=str(nf), label='NO_FACE', split='synthetic', crops_dir=''))

videos = pd.DataFrame(rows); videos.to_csv('/content/videos.csv', index=False)
print(videos[['video', 'label', 'split', 'crops_dir']].to_string(index=False))

In [ ]:
# 5. Training crops for these videos (from faces.tar, the exact JPEGs the model was trained/tested on)
import tarfile
FACES = Path('/content/faces_subset'); FACES.mkdir(exist_ok=True)
want = {d for d in videos.crops_dir if d}
n = 0
with tarfile.open(DRIVE_PROC / 'faces.tar') as tf:
    for mem in tf:
        name = mem.name[2:] if mem.name.startswith('./') else mem.name
        if mem.isfile() and str(Path(name).parent) in want:
            mem.name = name; tf.extract(mem, FACES); n += 1
print(n, 'training crops extracted for', len(want), 'videos')

In [ ]:
# 6. Run the validation (parity in-process, then the live API)
!cd /content && python parity_check.py --backend /content/repo/backend --model "{MODEL}" \
    --videos /content/videos.csv --faces /content/faces_subset --out "{OUT}" 2>&1 | grep -v Warning

In [ ]:
# 7. Results
import pandas as pd, json
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
parity = pd.read_csv(OUT / 'parity.csv'); api = pd.read_csv(OUT / 'api_results.csv')
print(json.load(open(OUT / 'summary.json')))
display(parity)
display(api)

print('frame indices match       :', parity.frame_indices_match.all())
print('detected flags match      :', (parity.detected_flags_match == True).all())
print('crops (32) identical      :', (parity.crops32_max_pixel_diff.fillna(0) == 0).all())
print('model tensor identical    :', (parity.tensor_max_diff.fillna(0) == 0).all())
print('sequence length 16        :', (parity.seq_len.dropna() == 16).all())
print('time order ok             :', parity.time_order_ok.dropna().astype(bool).all())
print('API == in-process verdict :', (api.verdict == api.inprocess_verdict).all())
print('Grad-CAM API == in-process:', (api.gradcam_vs_inprocess_mean_diff.dropna() == 0).all())
print('NO_FACE video -> NO_FACE  :', (api[api.expected == 'NO_FACE'].verdict == 'NO_FACE').all())

In [ ]:
# 8. Grad-CAM images returned by the API
import matplotlib.pyplot as plt
imgs = sorted(OUT.glob('gradcam_*.jpg'))
cols = 6; rows_ = (len(imgs) + cols - 1) // cols
fig, axes = plt.subplots(rows_, cols, figsize=(cols * 2.4, rows_ * 2.7))
for ax in axes.flat: ax.axis('off')
lab = dict(zip(api.video, api.verdict))
for ax, p in zip(axes.flat, imgs):
    name = p.stem.replace('gradcam_', '')
    ax.imshow(plt.imread(p)); ax.set_title(f'{name}\n{lab.get(name, "")}', fontsize=7)
plt.tight_layout(); plt.savefig(OUT / 'gradcam_grid.png', dpi=120); plt.show()

## What to post on PR #9
- `parity.csv`, `api_results.csv`, `summary.json`, `gradcam_grid.png` from `MyDrive/DeepGuard_runs/pr9_validation/`
- this notebook (with outputs), so the run can be repeated
- commit hash printed in step 3